# Wave Equation PINN — Colab Runner
Runs a single experiment (default: **PirateNet 3-block, soft IC, HomogeneousModel**) on a free Colab GPU.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Fill in `GITHUB_REPO_URL` in Cell 2
3. Run all cells top-to-bottom

In [ ]:
# ── Cell 1: GPU check ────────────────────────────────────────────────────────
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('VRAM  :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: no GPU — go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# ── Cell 2: Clone repo ───────────────────────────────────────────────────────
# Paste your private repo URL below (use HTTPS with a PAT, or make it public)
# Example: https://<PAT>@github.com/yourname/yourrepo.git
GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"

import os
if not os.path.isdir('Exp'):
    os.system(f'git clone {GITHUB_REPO_URL} Exp')
else:
    os.system('cd Exp && git pull')
print('Done.')

In [ ]:
# ── Cell 3: Install dependencies ─────────────────────────────────────────────
# PyTorch is pre-installed on Colab; just install the extra deps.
!pip install -q pykan==0.2.8 matplotlib numpy scipy

In [ ]:
# ── Cell 4: Configure experiment ─────────────────────────────────────────────
import sys, os
sys.path.insert(0, '/content/Exp')
sys.path.insert(0, '/content/Exp/wave')
os.chdir('/content/Exp')

# ---- What to run -------------------------------------------------------
# run_id 29 = PirateNet (n_blocks=3, width=64, sigma=3), HomogeneousModel, soft IC
# See wave/run_experiment.py MODEL_CONFIGS for other IDs.
RUN_ID = 29

# ---- Iteration budget --------------------------------------------------
# T4 is fast — these are the FULL production settings.
# Reduce if you want a quick smoke-test.
ADAM_ITERS          = 15000
LBFGS_ITERS         = 2000
CONTINUATION_BUDGET = 9000   # extra Adam steps if causal frontier stalls

OUTPUT_DIR = '/content/results/'

print(f'run_id={RUN_ID}  adam={ADAM_ITERS}  lbfgs={LBFGS_ITERS}  cont={CONTINUATION_BUDGET}')

In [ ]:
# ── Cell 5: Run experiment ───────────────────────────────────────────────────
import wave.run_experiment as exp
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Override CONFIG in-place before calling run_single_experiment
exp.CONFIG['adam_iterations']            = ADAM_ITERS
exp.CONFIG['lbfgs_iterations']           = LBFGS_ITERS
exp.CONFIG['causal_continuation_budget'] = CONTINUATION_BUDGET
exp.CONFIG['output_base_dir']            = OUTPUT_DIR

# Build the same run list the CLI uses, pick run RUN_ID
from wave.materials import HomogeneousModel, TwoLayerModel, MultiLayerModel
all_materials = [HomogeneousModel(), TwoLayerModel(), MultiLayerModel()]
adam_lbfgs_runs = [
    (mat, cfg, ansatz, False)
    for mat    in all_materials
    for cfg    in exp.MODEL_CONFIGS
    for ansatz in [True, False]
]
lbfgs_only_runs = [
    (mat, cfg, ansatz, True)
    for mat    in all_materials
    for cfg    in exp.MODEL_CONFIGS
    for ansatz in [True, False]
]
all_runs   = adam_lbfgs_runs + lbfgs_only_runs
total_runs = len(all_runs)

material, cfg, use_ansatz, lbfgs_only = all_runs[RUN_ID]
print(f'Running: {cfg["model_type"]} | material={material.name} | ansatz={use_ansatz}')

exp.run_single_experiment(
    cfg, use_ansatz, material, device,
    idx=RUN_ID + 1, total_runs=total_runs,
    lbfgs_only=lbfgs_only
)

In [ ]:
# ── Cell 6: Display results ──────────────────────────────────────────────────
import json, glob
from IPython.display import Image, display
import matplotlib.pyplot as plt

result_dirs = glob.glob(OUTPUT_DIR + '**/*.json', recursive=True)
for p in result_dirs:
    with open(p) as f:
        d = json.load(f)
    print('\n' + '='*60)
    print(p)
    for k, v in d.items():
        print(f'  {k}: {v}')

# Show plots
for png in glob.glob(OUTPUT_DIR + '**/*.png', recursive=True):
    print(f'\n--- {os.path.basename(png)} ---')
    display(Image(png))

In [ ]:
# ── Cell 7 (optional): Download results as zip ───────────────────────────────
from google.colab import files
import shutil

shutil.make_archive('/content/wave_results', 'zip', OUTPUT_DIR)
files.download('/content/wave_results.zip')